# Stage 2 v3 — Full Pipeline Augment Dataset

**Self-contained. Không phụ thuộc v1/v2.**

**Pipeline:**
1. Load `items_raw_tv_v6` + `items_tv_v6` → merge → push `items_tv_v7`
2. Filter price ≤ 1,000,000 VND → 85K items
3. Price bucket multiplier A5 (5x/3x/2x/1x/4x)
4. Groq Batch `gpt-oss-20b` → rewrite Mô tả + Thông số (~185K requests)
5. Parse outputs → push `items_tv_v8` (có cột `summary_version2`)
6. Combine 85K gốc + 185K aug → push `items_prompts_tv_4`

**Không cần GPU. Cần: `GROQ_API_KEY`, `HF_TOKEN`.**

In [1]:
import os
import re
import json
import time
import pickle
import random
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Optional

from datasets import load_dataset, DatasetDict, Dataset
from dotenv import load_dotenv
from huggingface_hub import login
from groq import Groq

NOTEBOOK_DIR = Path(".")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# --- HF datasets ---
SOURCE_RAW  = "SeanSunny/items_raw_tv_v6"
SOURCE_TV6  = "SeanSunny/items_tv_v6"
OUTPUT_TV7  = "SeanSunny/items_tv_v7"
OUTPUT_TV8  = "SeanSunny/items_tv_v8"
SOURCE_TV3  = "SeanSunny/items_prompts_tv_3"
OUTPUT_TV4  = "SeanSunny/items_prompts_tv_4"

MAX_PRICE     = 1_000_000
QUESTION_FULL = "Sản phẩm này có giá bao nhiêu ?"
PRICE_PREF    = "\n\nGiá là: "

# --- Folders (riêng để không xung đột v1/v2) ---
BATCHES_FOLDER = NOTEBOOK_DIR / "batches_aug_v3"
OUTPUT_FOLDER  = NOTEBOOK_DIR / "output_aug_v3"
STATE_FILE     = NOTEBOOK_DIR / "batches_aug_v3.pkl"
BATCHES_FOLDER.mkdir(parents=True, exist_ok=True)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# --- Env ---
env_path = NOTEBOOK_DIR.parent / ".env"
load_dotenv(env_path)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
GROQ_KEY = os.environ.get("GROQ_API_KEY", "")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN not set in .env")
if not GROQ_KEY:
    raise RuntimeError("GROQ_API_KEY not set in .env")

login(HF_TOKEN)
groq_client = Groq(api_key=GROQ_KEY)
print("HF login OK | Groq client OK | Imports OK")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login OK | Groq client OK | Imports OK


In [2]:
ds_v7 = load_dataset("SeanSunny/items_tv_v7")                                                                                                             
train_v7 = list(ds_v7["train"])                                                                                                                           
val_v7   = list(ds_v7["validation"])
test_v7  = list(ds_v7["test"])                                                                                                                            
print(f"Loaded: train={len(train_v7):,}") 

data/train-00000-of-00001.parquet:   0%|          | 0.00/134M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/6.04M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/6.09M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/110000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Loaded: train=110,000


## 1. Tạo items_tv_v7: merge full (raw) + summary (tv6)

In [2]:
print(f"Loading {SOURCE_RAW}...")
ds_raw = load_dataset(SOURCE_RAW)
print(f"  raw  train={len(ds_raw['train']):,} | val={len(ds_raw['validation']):,} | test={len(ds_raw['test']):,}")

print(f"Loading {SOURCE_TV6}...")
ds_tv6 = load_dataset(SOURCE_TV6)
print(f"  tv6  train={len(ds_tv6['train']):,} | val={len(ds_tv6['validation']):,} | test={len(ds_tv6['test']):,}")

# Verify alignment: title phải khớp theo từng vị trí
mismatches = sum(
    1 for i in range(min(1000, len(ds_raw['train'])))
    if ds_raw['train'][i]['title'] != ds_tv6['train'][i]['title']
)
print(f"Title alignment check (1000 samples): {mismatches} mismatches")
assert mismatches == 0, "Datasets not aligned — cannot merge by index!"
print("Alignment OK")

Loading SeanSunny/items_raw_tv_v6...
  raw  train=110,000 | val=5,000 | test=5,000
Loading SeanSunny/items_tv_v6...
  tv6  train=110,000 | val=5,000 | test=5,000
Title alignment check (1000 samples): 0 mismatches
Alignment OK


In [3]:
def merge_split(raw_split, tv6_split) -> list[dict]:
    """Merge: full từ raw, summary từ tv6. Giữ title/category/price/brand."""
    rows = []
    for raw_row, tv6_row in zip(raw_split, tv6_split):
        rows.append({
            "title":    raw_row["title"],
            "category": raw_row["category"],
            "price":    raw_row["price"],
            "full":     raw_row["full"],
            "brand":    raw_row.get("brand"),
            "summary":  tv6_row["summary"],
        })
    return rows

train_v7 = merge_split(ds_raw["train"],      ds_tv6["train"])
val_v7   = merge_split(ds_raw["validation"], ds_tv6["validation"])
test_v7  = merge_split(ds_raw["test"],       ds_tv6["test"])

print(f"Merged: train={len(train_v7):,} | val={len(val_v7):,} | test={len(test_v7):,}")

# Sanity check sample 0
s = train_v7[0]
assert s["full"],    "full is None — alignment issue!"
assert s["summary"], "summary is None — alignment issue!"
print(f"Sample 0 title  : {s['title'][:80]}")
print(f"Sample 0 full   : {len(s['full'])} chars")
print(f"Sample 0 summary: {s['summary'][:120]}")

Merged: train=110,000 | val=5,000 | test=5,000
Sample 0 title  : Pin Tương Thích Cho Laptop Dell Vostro 14 5459 - Hàng Nhập Khẩu New Seal TEEMO P
Sample 0 full   : 3104 chars
Sample 0 summary: Tiêu đề: Pin Tương Thích Dell Vostro 14 5459  
Danh mục: Pin Laptop  
Thương hiệu: TEEMO PC  
Mô tả: Pin Li‑ion mới, hoà


In [4]:
ds_v7 = DatasetDict({
    "train":      Dataset.from_list(train_v7),
    "validation": Dataset.from_list(val_v7),
    "test":       Dataset.from_list(test_v7),
})
print(f"Pushing {OUTPUT_TV7}...")
ds_v7.push_to_hub(OUTPUT_TV7, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV7}")

Pushing SeanSunny/items_tv_v7...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/110 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/674 [00:00<?, ?B/s]

No files have been modified since last commit. Skipping to prevent empty commit.


Pushed: https://huggingface.co/datasets/SeanSunny/items_tv_v7


## 2. Filter + parse_summary

In [5]:
train_filtered = [row for row in train_v7 if row["price"] <= MAX_PRICE]
print(f"Train after price filter: {len(train_filtered):,} / {len(train_v7):,}")
print(f"Dropped: {len(train_v7) - len(train_filtered):,} items (price > {MAX_PRICE:,} VND)")

# Gán index trong filtered list (dùng cho custom_id batch)
for idx, row in enumerate(train_filtered):
    row["_idx"] = idx


def parse_summary(summary: str):
    """Returns (header, body) or None. header=3 lines, body=2 lines."""
    if not summary:
        return None
    header_lines, body_lines = [], []
    for line in summary.strip().split("\n"):
        line = line.strip()
        if not line:
            continue
        if line.startswith("Tiêu đề:") or line.startswith("Tieu de:"):
            header_lines.append(line)
        elif line.startswith("Danh mục:") or line.startswith("Danh muc:"):
            header_lines.append(line)
        elif line.startswith("Thương hiệu:") or line.startswith("Thuong hieu:"):
            header_lines.append(line)
        elif line.startswith("Mô tả:") or line.startswith("Mo ta:"):
            body_lines.append(line)
        elif line.startswith("Thông số:") or line.startswith("Thong so:"):
            body_lines.append(line)
    if len(header_lines) == 3 and len(body_lines) == 2:
        return "\n".join(header_lines), "\n".join(body_lines)
    return None


# Verify trên 2000 samples
ok = fail = 0
for row in train_filtered[:2000]:
    if parse_summary(row["summary"]):
        ok += 1
    else:
        fail += 1
print(f"parse_summary check (2000 samples): OK={ok} FAIL={fail}")
if fail > 0:
    print("WARNING: some summaries have unexpected format!")
    for row in train_filtered[:2000]:
        if parse_summary(row["summary"]) is None:
            print(f"  FAIL: {repr(row['summary'][:200])}")
            break
else:
    h, b = parse_summary(train_filtered[0]["summary"])
    print(f"Sample header:\n{h}")
    print(f"Sample body:\n{b}")

Train after price filter: 85,727 / 110,000
Dropped: 24,273 items (price > 1,000,000 VND)
parse_summary check (2000 samples): OK=1998 FAIL=2
  FAIL: 'Tiêu đề: Giày Nam Da Bò Quai Ngang Cá Sấu  \nDanh mục: Giày Nam  \nThương thương hiệu: Giày Nam  \nMô tả: Giày cao cổ 100% da bò thật, quai ngang cá sấu mang phong cách lịch lãm, thẳng thắn.  \nThông số: '


## 3. Price bucket + multiplier A5

In [8]:
BUCKETS = [
    ("<50K",     0,          50_000,   5),
    ("50-100K",  50_000,    100_000,   3),
    ("100-200K", 100_000,   200_000,   2),
    ("200-500K", 200_000,   500_000,   1),
    ("500K-1M",  500_000, 1_000_001,   4),
]

prices = np.array([row["price"] for row in train_filtered])
bucket_multipliers = {}
total_aug = 0

print(f"{'Bucket':<12} {'Items':>7} {'%':>6} {'Mult':>5} {'Aug':>9}")
print("-" * 45)
for name, lo, hi, mult in BUCKETS:
    mask = (prices >= lo) & (prices < hi)
    count = int(mask.sum())
    aug = count * mult
    total_aug += aug
    print(f"{name:<12} {count:>7,} {count/len(prices)*100:>5.1f}%  {mult:>3}x  {aug:>8,}")
    for idx in np.where(mask)[0]:
        bucket_multipliers[int(idx)] = mult

print("-" * 45)
print(f"Total augmented rows : {total_aug:,}")
print(f"Total items_prompts_tv_4 train: {len(train_filtered) + total_aug:,} (85K orig + {total_aug:,} aug)")
assert len(bucket_multipliers) == len(train_filtered), "Some items missing multiplier!"
print("Multiplier assigned OK")

Bucket         Items      %  Mult       Aug
---------------------------------------------
<50K           1,285   1.5%    5x     6,425
50-100K       12,876  15.0%    3x    38,628
100-200K      23,939  27.9%    2x    47,878
200-500K      32,583  38.0%    1x    32,583
500K-1M       15,044  17.5%    4x    60,176
---------------------------------------------
Total augmented rows : 185,690
Total items_prompts_tv_4 train: 271,417 (85K orig + 185,690 aug)
Multiplier assigned OK


## 4. SYSTEM_PROMPT + AugBatchManager

In [9]:
MODEL      = "openai/gpt-oss-20b"
BATCH_SIZE = 1_000

SYSTEM_PROMPT_AUG = """Bạn là chuyên gia viết mô tả sản phẩm thương mại điện tử tiếng Việt.

Dựa vào thông tin sản phẩm gốc, hãy viết lại phần 'Mô tả' và 'Thông số' hiện tại bằng từ ngữ và cách diễn đạt khác.

Yêu cầu bắt buộc:
- Giữ nguyên toàn bộ số liệu và đơn vị đo (tuyệt đối không thay đổi bất kỳ con số nào).
- Diễn đạt tự nhiên, không sao chép cấu trúc câu gốc.
- Giữ đúng ngữ nghĩa và thông tin sản phẩm.
- Chỉ trả lời đúng 2 dòng theo định dạng sau, không thêm lời dẫn hay ký tự thừa:

Mô tả: [1 câu mô tả sản phẩm]
Thông số: [1 câu về tính năng hoặc thông số kỹ thuật nổi bật]"""


def build_user_message(full_text: str, body: str) -> str:
    return f"Thông tin sản phẩm gốc:\n{full_text}\n\n---\nTóm tắt hiện tại:\n{body}"


def parse_aug_output(llm_text: str):
    """Returns (mo_ta_text, thong_so_text) or None."""
    mo_ta = thong_so = None
    for line in llm_text.strip().split("\n"):
        line = line.strip()
        if line.startswith("Mô tả:") or line.startswith("Mo ta:"):
            mo_ta = re.sub(r'^(M[oô] t[aả]):?\s*', '', line).strip()
        elif line.startswith("Thông số:") or line.startswith("Thong so:"):
            thong_so = re.sub(r'^(Th[oô]ng s[oố]):?\s*', '', line).strip()
    if mo_ta and thong_so:
        return mo_ta, thong_so
    return None


@dataclass
class AugBatch:
    start: int
    end: int
    filename: str
    file_id: Optional[str] = None
    batch_id: Optional[str] = None
    output_file_id: Optional[str] = None
    done: bool = False


class AugBatchManager:
    batches: list = []
    request_list: list = []  # [(item_idx, version, user_msg), ...]

    @classmethod
    def build_requests(cls, filtered_items, multipliers):
        cls.request_list = []
        skipped = 0
        for idx, row in enumerate(tqdm(filtered_items, desc="Building requests")):
            result = parse_summary(row["summary"])
            if result is None:
                skipped += 1
                continue
            _, body = result
            full_text = row["full"] or ""
            for v in range(multipliers[idx]):
                cls.request_list.append((idx, v, build_user_message(full_text, body)))
        print(f"Requests built: {len(cls.request_list):,} | Skipped (bad summary): {skipped}")

    @classmethod
    def create_batches(cls):
        cls.batches = []
        for start in range(0, len(cls.request_list), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(cls.request_list))
            cls.batches.append(AugBatch(start, end, f"aug_{start}_{end}.jsonl"))
        print(f"Created {len(cls.batches)} batches ({BATCH_SIZE} requests each)")

    @classmethod
    def _make_jsonl_line(cls, item_idx, version, user_msg):
        return json.dumps({
            "custom_id": f"{item_idx}_{version}",
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_AUG},
                    {"role": "user",   "content": user_msg},
                ],
                "reasoning_effort": "low",
            },
        }, ensure_ascii=False)

    @classmethod
    def write_and_submit(cls, batch):
        fpath = BATCHES_FOLDER / batch.filename
        with fpath.open("w", encoding="utf-8") as f:
            for item_idx, v, user_msg in cls.request_list[batch.start:batch.end]:
                f.write(cls._make_jsonl_line(item_idx, v, user_msg) + "\n")
        with fpath.open("rb") as f:
            resp = groq_client.files.create(file=f, purpose="batch")
        batch.file_id = resp.id
        resp2 = groq_client.batches.create(
            completion_window="24h",
            endpoint="/v1/chat/completions",
            input_file_id=batch.file_id,
        )
        batch.batch_id = resp2.id

    @classmethod
    def run(cls):
        for batch in tqdm(cls.batches, desc="Submitting batches"):
            if not batch.batch_id:
                cls.write_and_submit(batch)
        print(f"Submitted {len(cls.batches)} batches")

    @classmethod
    def fetch(cls):
        for batch in cls.batches:
            if batch.done:
                continue
            result = groq_client.batches.retrieve(batch.batch_id)
            if result.status == "completed":
                batch.output_file_id = result.output_file_id
                groq_client.files.content(result.output_file_id).write_to_file(
                    str(OUTPUT_FOLDER / batch.filename)
                )
                batch.done = True
        finished = sum(1 for b in cls.batches if b.done)
        print(f"Finished {finished} / {len(cls.batches)} batches")
        return finished

    @classmethod
    def save(cls):
        with STATE_FILE.open("wb") as f:
            pickle.dump(cls.batches, f)
        print(f"State saved: {len(cls.batches)} batches → {STATE_FILE}")

    @classmethod
    def load(cls):
        with STATE_FILE.open("rb") as f:
            cls.batches = pickle.load(f)
        print(f"State loaded: {len(cls.batches)} batches")

    @classmethod
    def resubmit_failed(cls):
        resubmitted = 0
        for batch in cls.batches:
            if batch.done:
                continue
            result = groq_client.batches.retrieve(batch.batch_id)
            if result.status in ("failed", "expired", "cancelled"):
                cls.write_and_submit(batch)
                resubmitted += 1
                time.sleep(0.5)
        if resubmitted:
            cls.save()
        print(f"Resubmitted {resubmitted} batches")
        return resubmitted


print("AugBatchManager OK | MODEL:", MODEL)
print(f"SYSTEM_PROMPT_AUG (preview): {SYSTEM_PROMPT_AUG[:80]}...")

AugBatchManager OK | MODEL: openai/gpt-oss-20b
SYSTEM_PROMPT_AUG (preview): Bạn là chuyên gia viết mô tả sản phẩm thương mại điện tử tiếng Việt.

Dựa vào th...


In [10]:
print(f"SYSTEM_PROMPT_AUG (preview): {SYSTEM_PROMPT_AUG}...")

SYSTEM_PROMPT_AUG (preview): Bạn là chuyên gia viết mô tả sản phẩm thương mại điện tử tiếng Việt.

Dựa vào thông tin sản phẩm gốc, hãy viết lại phần 'Mô tả' và 'Thông số' hiện tại bằng từ ngữ và cách diễn đạt khác.

Yêu cầu bắt buộc:
- Giữ nguyên toàn bộ số liệu và đơn vị đo (tuyệt đối không thay đổi bất kỳ con số nào).
- Diễn đạt tự nhiên, không sao chép cấu trúc câu gốc.
- Giữ đúng ngữ nghĩa và thông tin sản phẩm.
- Chỉ trả lời đúng 2 dòng theo định dạng sau, không thêm lời dẫn hay ký tự thừa:

Mô tả: [1 câu mô tả sản phẩm]
Thông số: [1 câu về tính năng hoặc thông số kỹ thuật nổi bật]...


## 5. Test đơn lẻ — 3 items từ các bucket khác nhau

In [10]:
for tidx in [0, 1000, 50000]:
    row = train_filtered[tidx]
    result = parse_summary(row["summary"])
    if result is None:
        print(f"[{tidx}] SKIP: bad summary")
        continue
    header, body = result
    resp = groq_client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_AUG},
            {"role": "user",   "content": build_user_message(row["full"] or "", body)},
        ],
        reasoning_effort="low",
    )
    llm_out = resp.choices[0].message.content
    parsed = parse_aug_output(llm_out)
    print(f"\n[{tidx}] {row['title'][:65]} | {row['price']:,} VND")
    print(f"  ORIG body : {body.replace(chr(10), ' | ')}")
    if parsed:
        mo_ta, thong_so = parsed
        new_body = f"Mô tả: {mo_ta}\nThông số: {thong_so}"
        print(f"  NEW  body : {new_body.replace(chr(10), ' | ')}")
    else:
        print(f"  PARSE FAIL: {repr(llm_out[:100])}")
    print(f"  Tokens: {resp.usage.prompt_tokens}in / {resp.usage.completion_tokens}out")


[0] Áo len hoodie chất đẹp dày ấm thời trang trẻ trung cho nữ | 339,000 VND
  ORIG body : Mô tả: Hoodie chất dày, ấm áp, thiết kế trẻ trung, phối màu tươi sáng. | Thông số: Chất liệu len dày, mịn, tính năng ấm bảo vệ hoàn hảo cho mùa đông.
  NEW  body : Mô tả: Áo hoodie len dày, ấm áp, thiết kế trẻ trung, màu sắc tươi sáng phối màu xinh xắn. | Thông số: Chất liệu len dày, mịn, khả năng giữ nhiệt tuyệt vời, kích cỡ Freesize 60kg.
  Tokens: 586in / 80out

[1000] COMBO 10 QUẦN REN  CẠP CAO HÀNG VIỆT NAM SIÊU ĐẸP (40kg-78kg) | 210,000 VND
  ORIG body : Mô tả: Quần ren 3D mềm mịn, không viền, thoáng mát, thích hợp cho trang phục bó sát. | Thông số: Cân nặng phù hợp 40kg-78kg, đa dạng màu sắc tự nhiên, không lộ hay hằn mông.
  NEW  body : Mô tả: Quần ren 3D mịn, thoáng, không viền, vừa vặn với trang phục bó sát, tránh lộ và hằn mông. | Thông số: Dải cân từ 40kg đến 78kg, đa dạng màu sắc tự nhiên, sản phẩm được làm từ ren chất lượng cao.
  Tokens: 735in / 89out

[50000] (Loại Tốt) Ca Nấu Mì 

## 6. Test batch — 15 samples (Groq Batch API)

**[USER]** Kiểm tra chất lượng paraphrase. Nếu OK → chạy **Section 7 Full batch**.
Nếu cần chỉnh SYSTEM_PROMPT → sửa cell trên rồi chạy lại từ đây.

In [11]:
TEST_N = 15
TEST_IDXS = list(range(0, len(train_filtered), len(train_filtered) // TEST_N))[:TEST_N]

test_jsonl = BATCHES_FOLDER / "test_batch.jsonl"
test_reqs  = []

for idx in TEST_IDXS:
    row = train_filtered[idx]
    result = parse_summary(row["summary"])
    if result is None:
        continue
    _, body = result
    test_reqs.append((idx, 0, build_user_message(row["full"] or "", body)))

with test_jsonl.open("w", encoding="utf-8") as f:
    for item_idx, v, user_msg in test_reqs:
        f.write(AugBatchManager._make_jsonl_line(item_idx, v, user_msg) + "\n")

with test_jsonl.open("rb") as f:
    tf = groq_client.files.create(file=f, purpose="batch")
tb = groq_client.batches.create(
    completion_window="24h",
    endpoint="/v1/chat/completions",
    input_file_id=tf.id,
)
print(f"Test batch submitted: {tb.id} | Requests: {len(test_reqs)}")

Test batch submitted: batch_01kqbpzx84edp8tkzmmqev00dk | Requests: 15


In [12]:
test_out_path = OUTPUT_FOLDER / "test_batch.jsonl"
while True:
    r = groq_client.batches.retrieve(tb.id)
    if r.status == "completed":
        groq_client.files.content(r.output_file_id).write_to_file(str(test_out_path))
        print("Test batch DONE")
        break
    elif r.status in ("failed", "expired", "cancelled"):
        raise RuntimeError(f"Test batch {r.status}")
    print(f"Status: {r.status} — waiting 15s...")
    time.sleep(15)

Status: validating — waiting 15s...
Test batch DONE


In [13]:
# Hiển thị kết quả: ORIG body vs NEW body (summary_version2)
parse_fail = 0
print(f"=== Test Batch Results ({TEST_N} samples) ===\n")

with test_out_path.open(encoding="utf-8") as f:
    for line in f:
        obj      = json.loads(line)
        item_idx = int(obj["custom_id"].split("_")[0])
        llm_text = obj["response"]["body"]["choices"][0]["message"]["content"]
        row      = train_filtered[item_idx]
        header, orig_body = parse_summary(row["summary"])
        parsed   = parse_aug_output(llm_text)

        if parsed is None:
            parse_fail += 1
            print(f"[{item_idx}] PARSE FAIL: {repr(llm_text[:100])}")
            continue

        mo_ta, thong_so = parsed
        new_body = f"Mô tả: {mo_ta}\nThông số: {thong_so}"
        print(f"[{item_idx}] {row['title'][:55]} | {row['price']:,} VND")
        print(f"  ORIG: {orig_body.replace(chr(10), ' | ')}")
        print(f"  NEW : {new_body.replace(chr(10), ' | ')}")
        print()

print(f"Parse failures: {parse_fail}/{TEST_N}")
print("\n[USER] Kiểm tra chất lượng NEW body ở trên.")
print("       Nếu OK → chạy Section 7 Full batch.")
print("       Nếu cần chỉnh → sửa SYSTEM_PROMPT_AUG rồi chạy lại từ Section 6.")

=== Test Batch Results (15 samples) ===

[11430] Dung Dịch Vệ Sinh Miếng Dán Ngực SBeauty 50ml | 51,000 VND
  ORIG: Mô tả: Dung dịch khử khuẩn, làm sạch hiệu quả miếng dán ngực, giữ độ bền và vệ sinh lâu dài. | Thông số: Khử khuẩn lên tới 99%, làm sạch không gây kích ứng, giúp tăng tuổi thọ miếng dán từ 2–3 năm.
  NEW : Mô tả: Dung dịch vệ sinh miếng dán ngực SBeauty 50ml giúp làm sạch, khử khuẩn và loại bỏ mồ hôi, tăng độ bền miếng dán, tiết kiệm thời gian bảo quản. | Thông số: Khả năng khử khuẩn đạt 99%, làm sạch trong suốt không gây kích ứng da, kéo dài tuổi thọ miếng dán từ 2–3 năm.

[40005] Kính râm, Kính mát Mắt tròn nhỏ màu Hồng sành điệu | 169,000 VND
  ORIG: Mô tả: Kính râm thiết kế tròn nhỏ, màu hồng tinh tế, phù hợp với phong cách hiện đại. | Thông số: Chất liệu kim loại, tròng kính chống nắng và tia UV 100% hiệu quả.
  NEW : Mô tả: Kính râm mắt tròn màu hồng thanh lịch, kết hợp khung kim loại sang trọng, phù hợp với xu hướng thời trang hiện đại. | Thông số: Tròng kính bảo v

## 7. Full batch — ~185K requests

**[USER] Chỉ chạy sau khi confirm test batch OK.**

In [14]:
AugBatchManager.build_requests(train_filtered, bucket_multipliers)
AugBatchManager.create_batches()
print(f"\nEst. cost: ~{len(AugBatchManager.request_list)*750/1e6:.1f}M input tokens")
print(f"           @ ~$0.06/1M input + $0.60/1M output via Groq Batch")

Building requests:   0%|          | 0/85727 [00:00<?, ?it/s]

Requests built: 185,584 | Skipped (bad summary): 43
Created 186 batches (1000 requests each)

Est. cost: ~139.2M input tokens
           @ ~$0.06/1M input + $0.60/1M output via Groq Batch


In [15]:
AugBatchManager.run()

Submitting batches:   0%|          | 0/186 [00:00<?, ?it/s]

Submitted 186 batches


In [16]:
# QUAN TRỌNG: save ngay sau submit — nếu kernel crash sẽ mất batch_ids
AugBatchManager.save()

State saved: 186 batches → batches_aug_v3.pkl


In [17]:
# Poll cho đến khi xong (~1-6 giờ)
while True:
    finished = AugBatchManager.fetch()
    if finished == len(AugBatchManager.batches):
        print("All batches DONE!")
        break
    print(f"Waiting 60s... ({finished}/{len(AugBatchManager.batches)})")
    time.sleep(60)

Finished 2 / 186 batches
Waiting 60s... (2/186)
Finished 3 / 186 batches
Waiting 60s... (3/186)
Finished 4 / 186 batches
Waiting 60s... (4/186)
Finished 5 / 186 batches
Waiting 60s... (5/186)
Finished 5 / 186 batches
Waiting 60s... (5/186)
Finished 6 / 186 batches
Waiting 60s... (6/186)
Finished 7 / 186 batches
Waiting 60s... (7/186)
Finished 8 / 186 batches
Waiting 60s... (8/186)
Finished 8 / 186 batches
Waiting 60s... (8/186)
Finished 9 / 186 batches
Waiting 60s... (9/186)
Finished 10 / 186 batches
Waiting 60s... (10/186)
Finished 11 / 186 batches
Waiting 60s... (11/186)
Finished 11 / 186 batches
Waiting 60s... (11/186)
Finished 12 / 186 batches
Waiting 60s... (12/186)
Finished 13 / 186 batches
Waiting 60s... (13/186)
Finished 14 / 186 batches
Waiting 60s... (14/186)
Finished 14 / 186 batches
Waiting 60s... (14/186)
Finished 15 / 186 batches
Waiting 60s... (15/186)
Finished 16 / 186 batches
Waiting 60s... (16/186)
Finished 16 / 186 batches
Waiting 60s... (16/186)
Finished 17 / 186 ba

In [18]:
AugBatchManager.save()  # save final state

State saved: 186 batches → batches_aug_v3.pkl


### Resume (nếu kernel crash)
```python
# AugBatchManager.load()
# AugBatchManager.build_requests(train_filtered, bucket_multipliers)  # re-build request_list
# AugBatchManager.fetch()
```

In [ ]:
# Resubmit failed batches (nếu có)
#AugBatchManager.resubmit_failed()

## 8. Parse kết quả → build items_tv_v8

Mỗi `(item_idx, version)` → reconstruct `summary_version2 = header_gốc + new_body`.
`items_tv_v8` train = ~185K expanded rows, mỗi row là 1 augmented version của 1 item gốc.

In [19]:
# Đọc tất cả LLM outputs từ output folder
aug_results = {}  # (item_idx, version) -> llm_text

for batch in tqdm(AugBatchManager.batches, desc="Reading outputs"):
    out_path = OUTPUT_FOLDER / batch.filename
    if not out_path.exists():
        print(f"WARNING: missing {batch.filename}")
        continue
    with out_path.open(encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            cid = obj["custom_id"]
            item_idx, version = int(cid.split("_")[0]), int(cid.split("_")[1])
            aug_results[(item_idx, version)] = (
                obj["response"]["body"]["choices"][0]["message"]["content"]
            )

print(f"LLM outputs collected: {len(aug_results):,} / {len(AugBatchManager.request_list):,}")
print(f"Missing: {len(AugBatchManager.request_list) - len(aug_results)}")

Reading outputs:   0%|          | 0/186 [00:00<?, ?it/s]

LLM outputs collected: 185,583 / 185,584
Missing: 1


In [20]:
# Build items_tv_v8 rows
# Schema: tất cả cột của v7 + summary_version2 + aug_version
v8_rows = []
build_ok = build_fail = 0

for (item_idx, version), llm_text in tqdm(aug_results.items(), desc="Building v8 rows"):
    row    = train_filtered[item_idx]
    result = parse_summary(row["summary"])
    if result is None:
        build_fail += 1
        continue
    header, _ = result
    parsed = parse_aug_output(llm_text)
    if parsed is None:
        build_fail += 1
        continue
    mo_ta, thong_so = parsed
    new_body       = f"Mô tả: {mo_ta}\nThông số: {thong_so}"
    summary_v2     = f"{header}\n{new_body}"
    v8_rows.append({
        "title":            row["title"],
        "category":         row["category"],
        "price":            row["price"],
        "full":             row["full"],
        "brand":            row.get("brand"),
        "summary":          row["summary"],   # original từ v7
        "summary_version2": summary_v2,        # LLM rewritten
        "aug_version":      version,
    })
    build_ok += 1

print(f"v8 rows built: {build_ok:,} OK | {build_fail} failed")
if v8_rows:
    print(f"Sample summary_version2:\n{v8_rows[0]['summary_version2']}")

Building v8 rows:   0%|          | 0/185583 [00:00<?, ?it/s]

v8 rows built: 183,385 OK | 2198 failed
Sample summary_version2:
Tiêu đề: 40 Viên Pin Maxell AAA Than (Carbon)
Danh mục: Pin & Điện Lực
Thương hiệu: Maxell
Mô tả: Pin carbon Maxell AAA 40 viên, giá thành thấp, thích hợp cho đồ chơi, thiết bị gia đình và công nghệ.
Thông số: Dung tích 1,5V, kích thước 42mm x10mm, thời gian lưu trữ 3 năm.


## 9. Push items_tv_v8

In [ ]:
def add_v2_col(rows):
    return [{**row, "summary_version2": None, "aug_version": None} for row in rows]

ds_v8 = DatasetDict({
    "train":      Dataset.from_list(v8_rows),
    "validation": Dataset.from_list(add_v2_col(val_v7)),
    "test":       Dataset.from_list(add_v2_col(test_v7)),
})
print(ds_v8)
print(f"Train columns: {ds_v8['train'].column_names}")
print(f"Pushing {OUTPUT_TV8}...")
ds_v8.push_to_hub(OUTPUT_TV8, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV8}")

## 10. Build items_prompts_tv_4

- Train: 85K gốc (từ `items_prompts_tv_3`, dùng `summary` gốc) + ~185K aug (từ `items_tv_v8`, dùng `summary_version2`)
- Val / Test: giữ nguyên từ `items_prompts_tv_3`

In [ ]:
print(f"Loading {SOURCE_TV3}...")
ds_tv3 = load_dataset(SOURCE_TV3)
orig_train = list(ds_tv3["train"])
orig_val   = list(ds_tv3["val"])
orig_test  = list(ds_tv3["test"])
print(f"orig train={len(orig_train):,} | val={len(orig_val):,} | test={len(orig_test):,}")
assert set(orig_train[0].keys()) == {"prompt", "completion", "price_vnd_true"}
print("Schema OK")

In [ ]:
# Build augmented prompts từ items_tv_v8 summary_version2
aug_examples = []
for row in tqdm(v8_rows, desc="Building prompts tv4"):
    sv2 = row["summary_version2"]
    if not sv2:
        continue
    aug_examples.append({
        "prompt":         f"{QUESTION_FULL}\n{sv2}{PRICE_PREF}",
        "completion":     str(int(round(row["price"] / 1000))),
        "price_vnd_true": int(row["price"]),
    })

print(f"Augmented prompts built: {len(aug_examples):,}")

# Combine + shuffle
combined_train = orig_train + aug_examples
random.seed(SEED)
random.shuffle(combined_train)

print(f"Combined train: {len(orig_train):,} (orig) + {len(aug_examples):,} (aug) = {len(combined_train):,}")

# Quality check
empty_p = sum(1 for ex in combined_train if not ex["prompt"])
empty_c = sum(1 for ex in combined_train if not ex["completion"])
assert empty_p == 0 and empty_c == 0, f"Empty: {empty_p} prompts, {empty_c} completions"
prices_s = [ex["price_vnd_true"] for ex in combined_train]
print(f"Price range: {min(prices_s):,} — {max(prices_s):,} VND")
print("Quality check OK")

In [ ]:
ds_tv4 = DatasetDict({
    "train": Dataset.from_list(combined_train),
    "val":   Dataset.from_list(orig_val),
    "test":  Dataset.from_list(orig_test),
})
print(ds_tv4)
print(f"Pushing {OUTPUT_TV4}...")
ds_tv4.push_to_hub(OUTPUT_TV4, private=True)
print(f"Pushed: https://huggingface.co/datasets/{OUTPUT_TV4}")
print(f"\nDone! Train size: {len(combined_train):,} (~{len(combined_train)/1000:.0f}K)")

## Summary

| Dataset | Split | Rows | Note |
|---|---|---|---|
| `items_tv_v7` | train/val/test | 120K | raw merge: full + summary |
| `items_tv_v8` | train | ~185K | augmented rows, có cột `summary_version2` |
| `items_tv_v8` | val/test | 5K/5K | giữ v7, `summary_version2=None` |
| `items_prompts_tv_4` | train | ~270K | 85K orig + 185K aug |
| `items_prompts_tv_4` | val/test | 3,926/3,872 | giữ tv_3 |

**Bước tiếp:** Chạy `06_train_v4_scratch.ipynb` trên RTX 5090 32GB với `items_prompts_tv_4`.